# Notebook 04: Sales Intelligence
Notebook ini mencakup 10 seksi sesuai dengan Implementation Plan Day 4.

In [1]:
import sys
import os
import json
sys.path.append(os.path.abspath('..'))

from src.database import get_db, Product, Promotion
from src.sales_engine import SalesEngine
from src.conversation_manager import ConversationManager

engine = SalesEngine()
conv_manager = ConversationManager()
db = next(get_db())

promo = db.query(Promotion).filter_by(id="PROMO1").first()
if not promo:
    promo = Promotion(
        id="PROMO1",
        name="Diskon Akhir Pekan",
        discount_type="percentage",
        discount_value=10.0,
        active=True
    )
    db.add(promo)
    
snack = db.query(Product).filter_by(id="SNACK1").first()
if not snack:
    snack = Product(
        id="SNACK1",
        name="Snack Box Premium",
        category="snack",
        price=12000,
        minimum_order=10,
        active=True
    )
    db.add(snack)
    
db.commit()


## 4.1 Intent classification demo

In [2]:
msg1 = "Halo, mau nanya paket apa aja ya?"
msg2 = "Gimana cara pembayarannya?"
msg3 = "Saya mau pesan Paket A sekarang juga."

print(f"Pesan 1: {engine.analyze_message(msg1).intent}")
print(f"Pesan 2: {engine.analyze_message(msg2).intent}")
print(f"Pesan 3: {engine.analyze_message(msg3).intent}")


Pesan 1: product_inquiry
Pesan 2: other
Pesan 3: ordering


## 4.2 Budget detection (variasi bahasa)

In [3]:
msgs = [
    "Budget maksimal 25 ribu per kotak",
    "Ada ngga yang di bawah 30k?",
    "Dana kyknya cuma dua puluh ribu aja nih per porsi"
]
for m in msgs:
    print(f"'{m}' -> Budget: {engine.analyze_message(m).budget}")


'Budget maksimal 25 ribu per kotak' -> Budget: 25000.0
'Ada ngga yang di bawah 30k?' -> Budget: None
'Dana kyknya cuma dua puluh ribu aja nih per porsi' -> Budget: 20000.0


## 4.3 Quantity detection

In [4]:
msgs = [
    "Butuh 100 porsi",
    "Pesan 50 box ya",
    "Kira-kira buat lima ratus orang"
]
for m in msgs:
    print(f"'{m}' -> Qty: {engine.analyze_message(m).quantity}")


[INFO] TPM limit tercapai. Menunggu 36.4s...
'Butuh 100 porsi' -> Qty: 100
'Pesan 50 box ya' -> Qty: 50
'Kira-kira buat lima ratus orang' -> Qty: 500


## 4.4 Event type detection

In [5]:
msgs = [
    "Untuk acara nikahan minggu depan",
    "Buat meeting di kantor",
    "Acara pengajian rutin",
    "Buat Acara Keluarga"
]
for m in msgs:
    res = engine.analyze_message(m)
    print(f"'{m}' -> Event: {res.event_type}, Date: {res.event_date}")


[INFO] TPM limit tercapai. Menunggu 0.5s...
'Untuk acara nikahan minggu depan' -> Event: nikahan, Date: None
'Buat meeting di kantor' -> Event: meeting, Date: None
[INFO] TPM limit tercapai. Menunggu 6.6s...
[INFO] TPM limit tercapai. Menunggu 44.2s...
'Acara pengajian rutin' -> Event: pengajian rutin, Date: None
'Buat Acara Keluarga' -> Event: Keluarga, Date: None


## 4.5 Product recommendation engine

In [6]:
rekomendasi = engine.recommend_products(db, budget=25000, quantity=50, event_type="arisan")
for r in rekomendasi:
    print(f"- {r.name} (Rp{r.price})")


- Nasi Kotak Ayam Kampung (Rp24000.0)
- Nasi Kotak Broiler Jumbo (Rp23000.0)
- Nasi Kotak Broiler (Rp20000.0)


## 4.6 Price calculation (backend-computed)

In [7]:
if rekomendasi:
    calc = engine.calculate_price(db, product=rekomendasi[0], quantity=50)
    print(f"Produk: {rekomendasi[0].name}")
    print(f"Harga asli per box: Rp{calc.get('original_price_per_box', 0)}")
    print(json.dumps(calc, indent=2))


Produk: Nasi Kotak Ayam Kampung
Harga asli per box: Rp24000.0
{
  "original_price_per_box": 24000.0,
  "final_price_per_box": 22800.0,
  "base_total": 1200000.0,
  "discount_amount": 60000.0,
  "final_total": 1140000.0,
  "applied_promo": "Cashback 5% Corporate"
}


## 4.7 Upselling logic

In [8]:
upsells = engine.check_upsell(db, current_budget=23000, quantity=50)
for upsell in upsells:
        print(f"Peluang Upsell: Tawarkan {upsell.name} dengan harga Rp{upsell.price}")

Peluang Upsell: Tawarkan Nasi Kotak Ayam Kampung dengan harga Rp24000.0


## 4.8 Cross-selling logic

In [9]:
if rekomendasi:
    cross = engine.check_cross_sell(db, current_product=rekomendasi[0])
    print("Peluang Cross-sell:")
    for c in cross:
        print(f"- Tawarkan {c.name} (Rp{c.price}) sebagai pelengkap")


Peluang Cross-sell:
- Tawarkan Snack Box Standar (Rp10000.0) sebagai pelengkap
- Tawarkan Teh Kotak Sosro (Rp5000.0) sebagai pelengkap


## 4.9 Purchase intent tracking

In [10]:
msgs = [
    "Ada paket untuk acara kantor?",                       # LOW
    "Budget saya sekitar 25 ribu per box, ada?",           # MEDIUM
    "Kalau begitu saya ambil yang itu.",                   # HIGH
    "Catat pesanan saya 75 box untuk tanggal 20.",         # READY_TO_ORDER
]
for m in msgs:
    print(f"'{m}' -> Intent: {engine.analyze_message(m).purchase_intent}")


'Ada paket untuk acara kantor?' -> Intent: low
'Budget saya sekitar 25 ribu per box, ada?' -> Intent: medium
'Kalau begitu saya ambil yang itu.' -> Intent: ready_to_order
'Catat pesanan saya 75 box untuk tanggal 20.' -> Intent: medium


## 4.10 Multi-turn conversation demo

In [11]:
session_id = "demo_multi_turn"
chat = [
    "Halo, mau tanya katering untuk acara kawinan",
    "Rencana undang 500 orang",
    "Bisa kurang ngga kalau pesannya banyak? Budget 25rb per porsi"
]

for msg in chat:
    analysis = engine.analyze_message(msg)
    state = conv_manager.update_session(session_id, analysis)
    
print(json.dumps({k:v for k,v in state.items() if k != 'messages'}, indent=2))


[INFO] TPM limit tercapai. Menunggu 35.3s...
{
  "session_id": "demo_multi_turn",
  "quantity": 500,
  "budget_per_box": 25000.0,
  "event_type": null,
  "location": null,
  "event_date": null,
  "selected_product": null,
  "customer_name": null,
  "customer_phone": null,
  "purchase_intent": "MEDIUM"
}
